# Prompting Científico — Pipeline Verificable (Caso MalT)

**Capítulo 4 · Protocolos Multi-fase de Prompting Científico**

Este notebook implementa el protocolo de cinco fases como un **pipeline verificable**: cada fase tiene gates formales en código que impiden avanzar si la salida no cumple condiciones mínimas, y checkpoints donde el investigador revisa y aprueba antes de continuar.

### Diferencia con la versión básica

| Versión básica | Pipeline verificable |
|---|---|
| Prompts incompletos (sin 7 componentes) | Prompts canónicos (7 componentes) |
| Sin gates | Gates como condiciones Python |
| Revisión humana implícita | Checkpoints explícitos con `input()` |
| Sin log | Logger de sesión completo |
| Output → input sin validar | Validación de estructura antes de pasar a la siguiente fase |

### Antes de ejecutar

1. Descarga el paper: [Chapon (1982) — PMC553051](https://pmc.ncbi.nlm.nih.gov/articles/PMC553051/)
2. Configura tu API key: `export OPENAI_API_KEY="sk-..."`
3. Ejecuta las celdas secuencialmente — **no saltes celdas**.


## 0 · Instalación y configuración

In [ ]:
%pip install -q openai

In [ ]:
import os, json, re, datetime, textwrap
from pathlib import Path
from openai import OpenAI

client = OpenAI()   # Requiere OPENAI_API_KEY en el entorno

# Modelo — ajusta si usas otro proveedor compatible con la API de OpenAI
MODEL = "gpt-4o-mini"   # Cambia a "gpt-4o" para mayor precisión

print(f"Modelo seleccionado: {MODEL}")
print(f"Sesión iniciada: {datetime.datetime.now().isoformat()}")


## 1 · Logger de sesión y herramientas

El logger registra todos los prompts, respuestas, gates y decisiones humanas.  
Al final puedes exportarlo como JSON — es el equivalente al *Materials and Methods* de tu flujo LLM.


In [ ]:
class SessionLog:
    """Registra prompts, salidas y decisiones para trazabilidad completa."""

    def __init__(self, model: str):
        self.model = model
        self.started = datetime.datetime.now().isoformat()
        self.entries = []
        self.gates = {}
        self.tokens_total = 0

    def log(self, phase: str, event: str, content: str = "", metadata: dict = None):
        entry = {
            "ts": datetime.datetime.now().isoformat(),
            "phase": phase,
            "event": event,
            "content_preview": content[:300] + "…" if len(content) > 300 else content,
            "metadata": metadata or {}
        }
        self.entries.append(entry)

    def log_gate(self, phase: str, passed: bool, reason: str = ""):
        self.gates[phase] = {"passed": passed, "reason": reason}
        icon = "✓" if passed else "✗"
        print(f"  [{icon}] GATE {phase}: {reason}")

    def summary(self):
        passed = sum(1 for v in self.gates.values() if v["passed"])
        total  = len(self.gates)
        print(f"\nResumen de gates: {passed}/{total} pasaron")
        for k, v in self.gates.items():
            icon = "✓" if v["passed"] else "✗"
            print(f"  [{icon}] {k}: {v['reason']}")
        print(f"Tokens totales: {self.tokens_total}")

    def export(self, path: str = "session_log.json"):
        data = {
            "session_start": self.started,
            "session_end": datetime.datetime.now().isoformat(),
            "model": self.model,
            "tokens_total": self.tokens_total,
            "gates": self.gates,
            "entries": self.entries
        }
        Path(path).write_text(json.dumps(data, ensure_ascii=False, indent=2))
        print(f"Log exportado → {path}")
        return data


log = SessionLog(MODEL)
print("Logger inicializado.")


In [ ]:
def call_llm(prompt: str, phase: str = "", temperature: float = 0.1) -> str:
    """
    Llama al modelo. Temperatura baja (0.1) para reproducibilidad.
    Registra en el log de sesión.
    """
    log.log(phase, "PROMPT_SENT", prompt)

    response = client.chat.completions.create(
        model=MODEL,
        temperature=temperature,
        messages=[{"role": "user", "content": prompt}]
    )

    output = response.choices[0].message.content
    tokens = response.usage.total_tokens
    log.tokens_total += tokens
    log.log(phase, "OUTPUT_RECEIVED", output, {"tokens": tokens})

    return output


print("Función call_llm definida.")


## 2 · Gates epistémicos

Un **gate** es una condición Python que lanza `GateError` si no se cumple.  
A diferencia de un *checkpoint* (revisión humana), el gate es automático e impide que el pipeline continúe independientemente de la voluntad del usuario.


In [ ]:
class GateError(Exception):
    """Se lanza cuando un gate epistémico falla. Detiene el pipeline."""
    pass


def gate(condition: bool, phase: str, message: str):
    """
    Verifica una condición de gate.
    Si falla → GateError (la celda siguiente no puede ejecutarse).
    """
    if condition:
        log.log_gate(phase, True, message)
    else:
        log.log_gate(phase, False, message)
        raise GateError(
            f"\n{'='*60}\n"
            f"⛔  GATE {phase} NO PASÓ\n"
            f"{message}\n"
            f"{'='*60}\n"
            f"Corrige el problema antes de continuar."
        )


def parse_md_table(text: str) -> list[dict]:
    """
    Extrae la primera tabla Markdown del texto y devuelve lista de dicts.
    Devuelve [] si no hay tabla.
    """
    lines = text.splitlines()
    header, rows = None, []
    for i, line in enumerate(lines):
        if line.strip().startswith("|") and "---" not in line:
            cols = [c.strip() for c in line.strip().strip("|").split("|")]
            if header is None:
                header = cols
            else:
                rows.append(dict(zip(header, cols)))
    return rows


def checkpoint_humano(phase: str, instrucciones: str) -> None:
    """
    Pausa para revisión humana.
    Si el investigador rechaza, lanza GateError.
    """
    sep = "="*60
    print(f"\n{sep}\n🔍  CHECKPOINT HUMANO — {phase}\n{sep}")
    print(instrucciones)
    print()
    decision = input("¿Apruebas y continúas? [s/n]: ").strip().lower() or "s"
    if decision != "s":
        notas = input("Escribe qué hay que corregir: ")
        log.log(phase, "CHECKPOINT_RECHAZADO", notas)
        raise GateError(f"Investigador rechazó {phase}: {notas}")
    log.log(phase, "CHECKPOINT_APROBADO")
    print(f"  ✓ Checkpoint {phase} aprobado.\n")


print("Gates y checkpoints definidos.")


## 3 · Cargar el paper

El paper se carga desde el archivo de texto.  
Puedes reemplazar `paper_path` por la ruta a cualquier otro artículo en texto plano.


In [ ]:
# Ajusta la ruta si ejecutas el notebook desde otro directorio
paper_path = Path("6325162_PMC553051.txt")

if paper_path.exists():
    paper_text = paper_path.read_text(encoding="utf-8").strip()
    print(f"Paper cargado: {len(paper_text):,} caracteres")
    print(f"Primeras 300 caracteres:\n{paper_text[:300]}…")
else:
    # Fallback: pega el texto aquí si no tienes el archivo
    paper_text = """PEGA AQUÍ EL TEXTO COMPLETO DEL PAPER"""
    print("⚠ Archivo no encontrado. Usando texto manual.")

gate(len(paper_text) > 500, "CARGA",
     f"Texto del paper demasiado corto ({len(paper_text)} chars). ¿Cargaste el archivo correcto?")


---
## Fase 1 — Exploración

**Nivel de inferencia:** Bajo  
**Objetivo:** Localizar fragmentos donde MalT aparezca de forma explícita.  
**Lo que NO hace el modelo:** interpretar, resumir, ni inferir relaciones.


In [ ]:
PROMPT_F1 = f"""Rol: Explorador de contenido científico.

Contexto:
Texto completo del paper científico (Chapon, 1982) sobre expresión del gen malT
en Escherichia coli. Sin filtros previos.

Tarea:
Identifica párrafos del documento que mencionen explícitamente al menos una de las
siguientes palabras clave: MalT, regulación transcripcional, promotor, ensayo funcional,
Pribnow box, Shine-Dalgarno, malTp1, malTp7, expresión génica.

Alcance:
- Incluir solo fragmentos donde alguna de las palabras clave aparezca de forma explícita.
- Citar cada fragmento de forma textual y literal (párrafo completo, sin resumir ni truncar).
- Indicar el número de fragmento (F1, F2, …) y la sección del paper donde aparece.

Restricciones:
- No interpretar resultados.
- No resumir fragmentos.
- No inferir relaciones implícitas.
- No incluir fragmentos donde las palabras clave no aparezcan literalmente.

Salida esperada:
Para cada fragmento:
| ID | Sección | Fragmento literal | Palabras clave detectadas |

Verificación:
Confirmar que cada fragmento contiene al menos una palabra clave de forma explícita
y que la cita es literal e íntegra (no resumida).

Texto del paper:
{paper_text}
"""

print("Prompt Fase 1 preparado.")
print(f"Longitud del prompt: {len(PROMPT_F1):,} caracteres")


In [ ]:
print("Ejecutando Fase 1…")
f1_raw = call_llm(PROMPT_F1, phase="F1")
print("\n" + "─"*60)
print(f1_raw)


### Gate 1

Condiciones para avanzar:
1. El modelo encontró al menos 3 fragmentos.
2. Cada fragmento contiene al menos una palabra clave de forma explícita.
3. Ningún fragmento está vacío o tiene menos de 80 caracteres (indicaría que fue resumido).


In [ ]:
f1_rows = parse_md_table(f1_raw)

# Gate 1a: mínimo 3 fragmentos
gate(len(f1_rows) >= 3, "F1-cantidad",
     f"Se encontraron {len(f1_rows)} fragmentos. Se requieren al menos 3.")

# Gate 1b: todos tienen columna 'Fragmento literal' no vacía y suficientemente larga
fragmentos_cortos = [r for r in f1_rows
                     if len(r.get("Fragmento literal", "")) < 80]
gate(len(fragmentos_cortos) == 0, "F1-longitud",
     f"{len(fragmentos_cortos)} fragmentos con menos de 80 chars (posibles resúmenes): "
     + str([r.get("ID","?") for r in fragmentos_cortos]))

# Gate 1c: todos tienen palabras clave detectadas
sin_keywords = [r for r in f1_rows
                if not r.get("Palabras clave detectadas", "").strip()
                or r.get("Palabras clave detectadas", "").strip() == "—"]
gate(len(sin_keywords) == 0, "F1-keywords",
     f"{len(sin_keywords)} fragmentos sin palabras clave declaradas.")

print(f"\nFragmentos encontrados: {len(f1_rows)}")


In [ ]:
checkpoint_humano(
    phase="F1",
    instrucciones=(
        "Revisa la tabla anterior. Para cada fragmento verifica:\n"
        "  1. ¿El fragmento citado aparece literalmente en el paper?\n"
        "  2. ¿La palabra clave está explícita (no implícita)?\n"
        "  3. ¿Hay fragmentos relevantes que el modelo omitió?\n"
        "\nSi algún fragmento no cumple → responde 'n' y detalla qué corregir."
    )
)

# El investigador puede editar f1_aprobado antes de pasar a Fase 2
# Por defecto, se pasa la salida completa del modelo
f1_aprobado = f1_raw
print("Fase 1 completada. Fragmentos aprobados → entrada de Fase 2.")


---
## Fase 2 — Extracción literal estructurada

**Nivel de inferencia:** Bajo  
**Entrada:** Fragmentos aprobados por el investigador en Fase 1.  
**Salida:** Tabla estructurada con cita textual en cada fila.  
**Gate:** Ninguna fila sin cita textual; ningún campo inferido sin declararlo.


In [ ]:
PROMPT_F2 = f"""Rol: Extractor de información biológica de alta precisión.

Contexto:
Fragmentos aprobados por el investigador en Fase 1 del paper Chapon (1982).
No el paper completo.

Tarea:
Extrae únicamente información explícita sobre las relaciones regulatorias presentes
en los fragmentos. Para cada relación identificada, completa una fila de la tabla
de salida usando solo información presente de forma explícita en el texto.

Alcance:
- Incluir únicamente información presente de forma explícita en los fragmentos.
- Cada fila debe incluir cita textual exacta del fragmento de origen (no parafrasear).
- En "Regulador": incluir solo proteínas, complejos o ligandos (MalT, CAP, cAMP).
  Las mutaciones (malTp1, malTp7) van en "Condición experimental".
- Si un campo no está especificado en el fragmento, escribir exactamente: No especificado.

Restricciones:
- No inferir relaciones no explícitas en el texto.
- No completar información ausente con conocimiento propio.
- No parafrasear la cita textual.
- No incluir filas sin respaldo literal en los fragmentos aprobados.

Salida esperada:
| Regulador | Gen regulado | Evidencia | Condición experimental | Resultado reportado | Cita textual |

Verificación:
Confirmar que cada fila tiene cita textual no vacía y que el campo "Regulador"
no contiene mutaciones (esas van en "Condición experimental").

Fragmentos aprobados (Fase 1):
{f1_aprobado}
"""

print("Prompt Fase 2 preparado.")


In [ ]:
print("Ejecutando Fase 2…")
f2_raw = call_llm(PROMPT_F2, phase="F2")
print("\n" + "─"*60)
print(f2_raw)


### Gate 2

Condiciones para avanzar:
1. La salida contiene una tabla Markdown con al menos 2 filas de datos.
2. Todas las columnas requeridas están presentes.
3. Ninguna fila tiene "Cita textual" vacía.
4. El campo "Regulador" no contiene mutaciones (malTp1, malTp7).


In [ ]:
f2_rows = parse_md_table(f2_raw)

# Gate 2a: mínimo 2 filas
gate(len(f2_rows) >= 2, "F2-cantidad",
     f"La tabla tiene {len(f2_rows)} filas. Se requieren al menos 2.")

# Gate 2b: columnas requeridas
required_cols = {"Regulador", "Gen regulado", "Cita textual"}
existing_cols = set(f2_rows[0].keys()) if f2_rows else set()
missing_cols = required_cols - existing_cols
gate(len(missing_cols) == 0, "F2-columnas",
     f"Columnas requeridas ausentes: {missing_cols}")

# Gate 2c: ninguna fila sin cita textual
sin_cita = [i+1 for i, r in enumerate(f2_rows)
            if not r.get("Cita textual","").strip()
            or r.get("Cita textual","").strip() in ("", "—", "-")]
gate(len(sin_cita) == 0, "F2-citas",
     f"Filas sin cita textual: {sin_cita}. Toda fila requiere respaldo literal.")

# Gate 2d: Regulador no contiene mutaciones
MUTACIONES = ["malTp1", "malTp7", "malTp", "malT1", "malT7"]
reguladores_con_mutacion = [
    (i+1, r.get("Regulador",""))
    for i, r in enumerate(f2_rows)
    if any(m in r.get("Regulador","") for m in MUTACIONES)
]
gate(len(reguladores_con_mutacion) == 0, "F2-regulador",
     f"Fila(s) con mutación en columna Regulador: {reguladores_con_mutacion}. "
     f"Las mutaciones van en 'Condición experimental'.")

print(f"\nFilas extraídas: {len(f2_rows)}")


In [ ]:
checkpoint_humano(
    phase="F2",
    instrucciones=(
        "Revisa la tabla de Fase 2. Para cada fila verifica:\n"
        "  1. ¿La cita textual aparece verbatim en el paper?\n"
        "  2. ¿La columna 'Regulador' contiene solo proteínas/ligandos (no mutaciones)?\n"
        "  3. ¿Algún campo está inferido sin declararse como 'No especificado'?\n"
        "\nPuedes editar la variable f2_aprobado antes de continuar."
    )
)

f2_aprobado = f2_raw
print("Fase 2 completada. Tabla aprobada → entrada de Fase 3.")


---
## Fase 3 — Normalización y curación

**Nivel de inferencia:** Medio  
**Entrada:** Tabla validada de Fase 2.  
**Salida:** Tabla extendida con nombres canónicos, accessions y ambigüedades declaradas.  
**Gate:** Nombres originales preservados; equivalencias trazables al texto fuente.


In [ ]:
PROMPT_F3 = f"""Rol: Curador bioinformático de alta precisión.

Contexto:
Tabla de Fase 2 validada por el investigador. Paper: Chapon (1982),
entidades del regulón malT de Escherichia coli.

Tarea:
Estandariza los nombres génicos de la tabla usando nomenclatura oficial;
diferencia gen (malT), proteína (MalT) y ligando (cAMP); reporta la equivalencia
con el nombre del paper; asigna identificador único cuando sea posible
(EcoCyc, UniProtKB, ChEBI); e indica explícitamente cualquier ambigüedad.

Alcance:
- Conservar todas las columnas de Fase 2 en la salida (trazabilidad acumulativa).
- Añadir columnas: Nombre estandarizado | Tipo (gen/proteína/ligando) | Accession | Fuente | Ambigüedad (Sí/No) | Comentario.
- Incluir la fuente de cada equivalencia reportada (EcoCyc, UniProt, ChEBI, etc.).
- Comentar todos los casos de ambigüedad o correspondencia múltiple.

Restricciones:
- No eliminar el nombre original del paper (columna "Regulador" / "Gen regulado").
- No asumir equivalencias no verificables con una base de datos externa.
- Si no hay identificador con certeza: escribir exactamente VERIFICACIÓN MANUAL REQUERIDA.
- No borrar columnas de Fase 2.

Salida esperada:
| Regulador | Gen regulado | Evidencia | Condición experimental | Resultado reportado | Cita textual | Nombre estandarizado | Tipo | Accession | Fuente | Ambigüedad | Comentario |

Verificación:
Confirmar que el nombre original del paper está preservado y que cada equivalencia
es trazable al texto fuente. Los accessions con dudas deben marcarse como
VERIFICACIÓN MANUAL REQUERIDA, no omitirse.

Tabla Fase 2 (entrada):
{f2_aprobado}
"""

print("Prompt Fase 3 preparado.")


In [ ]:
print("Ejecutando Fase 3…")
f3_raw = call_llm(PROMPT_F3, phase="F3")
print("\n" + "─"*60)
print(f3_raw)


### Gate 3

Condiciones para avanzar:
1. Las columnas de Fase 2 siguen presentes (trazabilidad acumulativa).
2. Existen las columnas nuevas de normalización.
3. Ningún nombre original eliminado.
4. Entradas con `VERIFICACIÓN MANUAL REQUERIDA` flaggeadas para revisión.


In [ ]:
f3_rows = parse_md_table(f3_raw)

# Gate 3a: columnas de Fase 2 preservadas
f2_cols = set(f2_rows[0].keys()) if f2_rows else set()
f3_cols = set(f3_rows[0].keys()) if f3_rows else set()
cols_perdidas = f2_cols - f3_cols
gate(len(cols_perdidas) == 0, "F3-trazabilidad",
     f"Columnas de Fase 2 perdidas en Fase 3: {cols_perdidas}. "
     f"La tabla debe ser acumulativa.")

# Gate 3b: columnas nuevas requeridas
nuevas_requeridas = {"Nombre estandarizado", "Accession", "Ambigüedad"}
nuevas_presentes = nuevas_requeridas & f3_cols
gate(len(nuevas_presentes) == len(nuevas_requeridas), "F3-columnas-nuevas",
     f"Columnas de normalización ausentes: {nuevas_requeridas - nuevas_presentes}")

# Gate 3c: cantidad de filas conservada
gate(len(f3_rows) >= len(f2_rows), "F3-filas",
     f"Fase 3 tiene {len(f3_rows)} filas; Fase 2 tenía {len(f2_rows)}. "
     f"No se pueden eliminar filas.")

# Informe de entradas con verificación manual requerida
vmr = [i+1 for i, r in enumerate(f3_rows)
       if "VERIFICACIÓN MANUAL REQUERIDA" in r.get("Accession","").upper()
       or "VERIFICACIÓN MANUAL REQUERIDA" in r.get("Comentario","").upper()]
if vmr:
    print(f"\n⚠  Filas que requieren verificación manual: {vmr}")
    print("   El investigador debe resolver estas antes de Fase 4.")

print(f"\nFilas normalizadas: {len(f3_rows)}")


In [ ]:
checkpoint_humano(
    phase="F3",
    instrucciones=(
        "Revisa la tabla de Fase 3. Verifica:\n"
        "  1. ¿El nombre original del paper está preservado en cada fila?\n"
        "  2. ¿Cada equivalencia puede trazarse al texto del paper (no es inferencia nueva)?\n"
        "  3. ¿Las entradas con VERIFICACIÓN MANUAL REQUERIDA son correctas?\n"
        "     (Abre EcoCyc/UniProt y confirma manualmente antes de aprobar.)\n"
        "  4. ¿La ambigüedad malK-lamB está resuelta o marcada?\n"
        "\nEdita f3_aprobado si necesitas ajustar antes de continuar."
    )
)

f3_aprobado = f3_raw
print("Fase 3 completada. Tabla curada → entrada de Fase 4.")


---
## Fase 4 — Evaluación crítica de evidencia

**Nivel de inferencia:** Alto  
**Entrada:** Tabla normalizada de Fase 3 (con ambigüedades resueltas).  
**Salida:** Tabla con clasificación de evidencia + justificación textual.  
**Gate 4 es OBLIGATORIO** — ninguna clasificación se acepta sin revisión humana explícita.


In [ ]:
PROMPT_F4 = f"""Rol: Revisor metodológico.

Contexto:
Tabla normalizada de Fase 3, con ambigüedades de nomenclatura resueltas por el investigador.
Paper: Chapon (1982) sobre expresión de malT en E. coli.

Tarea:
Clasifica cada evidencia experimental como Directa, Indirecta o Correlacional.
Define un criterio operativo explícito (máx. 1 línea) para cada categoría antes de clasificar.

Alcance:
- Incluir definición operativa de cada categoría al inicio de la salida (antes de la tabla).
- Mantener exactamente las mismas filas y columnas de la tabla de entrada.
- Añadir columnas: Clasificación | Criterio aplicado | Justificación textual | Limitaciones detectadas.

Restricciones:
- Trabajar exclusivamente con las filas de la tabla de Fase 3.
- No incorporar evidencia nueva del artículo ni de fuentes externas.
- La justificación debe derivarse de la cita textual ya presente en la tabla.
- Si una entrada no permite clasificación clara: escribir exactamente No clasificable + motivo.
- No asumir robustez metodológica sin evidencia explícita en la cita.

Salida esperada:
Primero, las definiciones operativas:
  - Directa: …
  - Indirecta: …
  - Correlacional: …

Luego la tabla:
| [columnas de F3] | Clasificación | Criterio aplicado | Justificación textual | Limitaciones detectadas |

Verificación:
Revisión humana obligatoria de cada clasificación y su correspondencia con la cita.
Esta fase no puede aceptarse sin aprobación explícita del investigador.

Tabla Fase 3 (entrada):
{f3_aprobado}
"""

print("Prompt Fase 4 preparado.")


In [ ]:
print("Ejecutando Fase 4…")
f4_raw = call_llm(PROMPT_F4, phase="F4", temperature=0.2)
print("\n" + "─"*60)
print(f4_raw)


### Gate 4 — Verificación estructural (automática)

Este gate verifica solo la *estructura* de la salida.  
La verificación del *contenido* (si las clasificaciones son correctas) es responsabilidad exclusiva del investigador en el checkpoint siguiente.


In [ ]:
f4_rows = parse_md_table(f4_raw)

# Gate 4a: columnas de clasificación presentes
cols_f4 = set(f4_rows[0].keys()) if f4_rows else set()
clasificacion_cols = {"Clasificación", "Justificación textual"}
faltantes = clasificacion_cols - cols_f4
gate(len(faltantes) == 0, "F4-columnas",
     f"Columnas de clasificación ausentes: {faltantes}")

# Gate 4b: ninguna clasificación vacía
sin_clasificacion = [i+1 for i, r in enumerate(f4_rows)
                     if not r.get("Clasificación","").strip()
                     or r.get("Clasificación","").strip() in ("", "—")]
gate(len(sin_clasificacion) == 0, "F4-clasificaciones",
     f"Filas sin clasificación: {sin_clasificacion}")

# Gate 4c: ninguna justificación vacía
sin_justificacion = [i+1 for i, r in enumerate(f4_rows)
                     if not r.get("Justificación textual","").strip()
                     or r.get("Justificación textual","").strip() in ("", "—")]
gate(len(sin_justificacion) == 0, "F4-justificaciones",
     f"Filas sin justificación textual: {sin_justificacion}")

# Conteo de categorías
from collections import Counter
cats = Counter(r.get("Clasificación","").strip() for r in f4_rows)
print("\nDistribución de clasificaciones:")
for cat, n in sorted(cats.items()):
    print(f"  {cat}: {n}")


In [ ]:
# ⚠ CHECKPOINT OBLIGATORIO — no hay override posible
print("\n" + "⚠"*30)
print("REVISIÓN HUMANA OBLIGATORIA — FASE 4")
print("⚠"*30)
print("""
Esta fase tiene nivel de inferencia ALTO.
El modelo aplicó criterios interpretativos que pueden ser incorrectos.

Para cada fila de la tabla, debes verificar:

  1. ¿Los criterios operativos (Directa / Indirecta / Correlacional)
     son los que tú usarías para este tipo de evidencia?

  2. ¿La clasificación de cada fila es consistente con la cita textual?

  3. ¿Hay filas que deberían ser "No clasificable" y no lo son?

  4. ¿Hay limitaciones metodológicas reales que el modelo no detectó?

Si cambiaste alguna clasificación, edita f4_aprobado con la versión corregida.
""")

checkpoint_humano(
    phase="F4-OBLIGATORIO",
    instrucciones="Confirma haber revisado CADA fila individualmente."
)

f4_aprobado = f4_raw
print("Fase 4 completada. Clasificaciones aprobadas por el investigador.")


---
## Fase 5 — Integración y juicio científico

> **Esta fase no se delega al modelo.**

El investigador escribe aquí sus conclusiones a partir de las cuatro fases anteriores.  
El notebook proporciona una plantilla y acceso a todas las salidas aprobadas.


In [ ]:
# ── Plantilla para el investigador ──────────────────────────────────────────
# Completa las secciones entre comillas con tu propio análisis.

INTEGRACION_HUMANA = """
## Síntesis — Regulación de malT en E. coli (Chapon, 1982)

### 1. Pregunta de investigación respondida
[¿Qué evidencia encontramos sobre la regulación de malT?]

### 2. Hallazgos principales (de Fase 4)
[Enumera las relaciones regulatorias con mayor solidez metodológica.]

### 3. Calidad de la evidencia
- Evidencias directas: [n] — [descripción breve]
- Evidencias indirectas: [n] — [descripción breve]
- Evidencias correlacionales: [n] — [descripción breve]

### 4. Limitaciones metodológicas identificadas
[¿Qué ensayos son de 1982 y podrían no reflejar el estado del arte?
¿Qué afirmaciones del paper no pudieron clasificarse?]

### 5. Incertidumbre declarada
[¿Qué preguntas quedan abiertas? ¿Qué requeriría validación experimental adicional?]

### 6. Decisión: ¿Es este paper suficiente para el objetivo del análisis?
[Sí/No/Parcialmente — justificación en 2-3 líneas]
"""

print("Plantilla de Fase 5 lista.")
print("\nEdita la variable INTEGRACION_HUMANA con tu análisis antes de exportar.")


In [ ]:
# Vista previa de la integración
print(INTEGRACION_HUMANA)


---
## Trazabilidad — Exportar log de sesión

El log registra: modelo usado, temperatura, todos los prompts enviados, todas las respuestas recibidas, resultado de cada gate y decisión de cada checkpoint.

Es el equivalente al *Materials and Methods* de un flujo LLM reproducible.


In [ ]:
# Añadir la integración humana al log
log.log("F5", "INTEGRACION_HUMANA", INTEGRACION_HUMANA)

# Resumen en pantalla
log.summary()

# Exportar JSON
export_path = f"session_log_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
log.export(export_path)

print(f"\nEl log puede citarse en tu sección de Métodos como:")
print(f"  'El análisis se realizó con el modelo {MODEL}, temperatura 0.1,")
print(f"   protocolo de cinco fases con gates epistémicos verificables'")
print(f"  'Los prompts y salidas completos están disponibles en: {export_path}'")


---
## Referencias

- Chapon, C. (1982). Expression of *malT*, the regulator gene of the maltose region in *Escherichia coli*, is limited both at transcription and translation. *The EMBO Journal*, 1(3), 369–374. PMID: 6325162
- Brown, T. B. et al. (2020). Language Models are Few-Shot Learners. *NeurIPS*.
- Ji, Z. et al. (2023). Survey of Hallucination in Natural Language Generation. *ACM Computing Surveys*.

---
*Notebook generado para el Capítulo 4 — Protocolos Multi-fase de Prompting Científico.*
